# Metaheuristic Optimization Lab (Lecture 6 — Heuristic/Metaheuristic)

เล่มนี้ต่อจาก `optimization_lecture6_lab.ipynb` ตรงส่วน **Heuristic/Metaheuristic** ที่ก่อนหน้านี้มีแค่ตารางสรุป
— ในไฟล์นี้จะ **implement จริงตั้งแต่ต้น (ไม่ใช้ optimization library)** ตามที่ Assignment 1 กำหนด:

1. **Simulated Annealing (SA)** — single-solution / trajectory-based
2. **Particle Swarm Optimization (PSO)** — population-based
3. ทดสอบทั้งคู่บน **Rosenbrock function** และ **Rastrigin function**
4. แสดงการคำนวณด้วยมืออย่างน้อย 2 iterations (ตามที่โจทย์กำหนด)
5. รันจริงอย่างน้อย 20 iterations พร้อม Best/Mean/Worst Fitness, Average Runtime, Convergence Curve
6. เปรียบเทียบ SA vs PSO และตัวอย่างการนำไปใช้จริง

**หมายเหตุ:** โค้ดทั้งหมดเขียนขึ้นเองด้วย `numpy` ล้วนๆ (ไม่ใช้ `scipy.optimize`, `deap`, หรือ optimization library ใดๆ)
ตรงตามเงื่อนไข "ห้ามใช้ Optimization Lib" ของ assignment


In [ ]:
import numpy as np
import pandas as pd
import time
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.precision', 4)


## 1. Benchmark functions

$$\text{Rosenbrock: } f(x,y) = (1-x)^2 + 100(y-x^2)^2 \qquad \text{Global min: } f(1,1)=0$$

$$\text{Rastrigin: } f(x,y) = 20 + x^2+y^2 - 10[\cos(2\pi x)+\cos(2\pi y)] \qquad \text{Global min: } f(0,0)=0$$

**ความแตกต่างสำคัญ (เชื่อมกับ Lecture 5 เรื่อง Convex/Non-convex):**

- **Rosenbrock** — มี minimum จุดเดียว แต่อยู่ใน **หุบเขาโค้งแคบยาว** (banana-shaped valley) ทำให้ gradient-based method ลู่เข้าช้ามาก แม้จะเป็น unimodal
- **Rastrigin** — มี **local minima นับร้อยจุด** กระจายเป็นตาราง (เพราะพจน์ $\cos$) ทำให้ metaheuristic ติดหลุมง่ายมาก — เป็นตัวทดสอบมาตรฐานสำหรับวัด "ความสามารถหนีหลุม" ของอัลกอริทึม


In [ ]:
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2


def rastrigin(x, y):
    return 20 + x**2 + y**2 - 10 * (np.cos(2*np.pi*x) + np.cos(2*np.pi*y))


def plot_function_3d(func, xr, yr, title, log_z=False):
    xs = np.linspace(*xr, 100)
    ys = np.linspace(*yr, 100)
    X, Y = np.meshgrid(xs, ys)
    Z = func(X, Y)
    Zplot = np.log10(Z + 1) if log_z else Z

    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Zplot, colorscale="Viridis", showscale=False)])
    fig.update_layout(title=title + (" (log scale)" if log_z else ""),
                       scene={"xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "f(x,y)"},
                       width=550, height=480, margin={"l": 0, "r": 0, "t": 40, "b": 0})
    fig.show()

plot_function_3d(rosenbrock, (-2, 2), (-1, 3), "Rosenbrock function", log_z=True)
plot_function_3d(rastrigin, (-5, 5), (-5, 5), "Rastrigin function")


## 2. Simulated Annealing (SA)

**แรงบันดาลใจ:** เลียนแบบกระบวนการ **annealing ทางโลหะวิทยา** — โลหะร้อนจัดมีอะตอมเคลื่อนที่อิสระ (สำรวจกว้าง)
พอค่อยๆ เย็นลง อะตอมจะ "ล็อกตัว" เข้าโครงสร้างพลังงานต่ำ — ในทางเดียวกัน อัลกอริทึมยอมรับคำตอบที่ **แย่กว่า** ได้ในช่วง
"อุณหภูมิ" $T$ สูง (เพื่อหนี local minimum) แล้วค่อยๆ ลด $T$ ให้ยอมรับคำตอบแย่ยากขึ้นเรื่อยๆ

**ขั้นตอนอัลกอริทึม**

1. เริ่มจากจุดสุ่ม $x$ และอุณหภูมิเริ่มต้น $T_0$
2. สุ่มจุดข้างเคียง (neighbor) $x' = x + \mathcal{N}(0,\text{step})$
3. คำนวณ $\Delta = f(x')-f(x)$
   - ถ้า $\Delta<0$ (ดีขึ้น) → **ยอมรับเสมอ**
   - ถ้า $\Delta\ge0$ (แย่ลง) → ยอมรับด้วยความน่าจะเป็น $P=e^{-\Delta/T}$ (Metropolis criterion)
4. ลดอุณหภูมิ: $T \leftarrow T\times\text{cooling rate}$ (เช่น $\times0.9$ หรือ $\times0.995$)
5. ทำซ้ำจนครบจำนวนรอบ แล้วคืนค่าคำตอบที่ดีที่สุดที่เจอ (`best`, แยกจาก `current` ที่อาจแย่กว่าเพราะยอมรับ probabilistic)

**สมการสำคัญ:** $\;P(\text{accept})=\exp\!\left(-\dfrac{\Delta}{T}\right)$ — $T$ สูง → $P$ ใกล้ 1 (ยอมรับง่าย); $T$ ต่ำ → $P$ ใกล้ 0 (ยอมรับยาก)


In [ ]:
def simulated_annealing(func, bounds, n_iter=20, T0=10.0, cooling=0.9, step=0.5, seed=None, verbose=False):
    """
    func    : ฟังก์ชันเป้าหมาย f(x, y) ที่ต้อง minimize
    bounds  : [(xlo, xhi), (ylo, yhi)]
    n_iter  : จำนวนรอบ
    T0      : อุณหภูมิเริ่มต้น
    cooling : อัตราลดอุณหภูมิต่อรอบ (0 < cooling < 1)
    step    : ส่วนเบี่ยงเบนมาตรฐานของการสุ่ม neighbor
    """
    rng = np.random.default_rng(seed)
    (xlo, xhi), (ylo, yhi) = bounds
    x = rng.uniform(xlo, xhi)
    y = rng.uniform(ylo, yhi)
    f_curr = func(x, y)
    best_x, best_y, best_f = x, y, f_curr
    T = T0
    history = [best_f]

    for it in range(1, n_iter + 1):
        x_new = np.clip(x + rng.normal(0, step), xlo, xhi)
        y_new = np.clip(y + rng.normal(0, step), ylo, yhi)
        f_new = func(x_new, y_new)
        delta = f_new - f_curr
        accept = (delta < 0) or (rng.random() < np.exp(-delta / T))

        if verbose and it <= 2:
            if delta < 0:
                acc_str = "delta<0 -> ยอมรับทันที (ดีขึ้น)"
            else:
                acc_str = f"P=exp(-{delta:.4f}/{T:.4f})={np.exp(-delta/T):.4f} -> accept={accept}"
            print(f"Iter {it}: current=({x:.4f},{y:.4f}) f={f_curr:.4f}")
            print(f"         candidate=({x_new:.4f},{y_new:.4f}) f={f_new:.4f}  delta={delta:.4f}")
            print(f"         T={T:.4f}  {acc_str}\n")

        if accept:
            x, y, f_curr = x_new, y_new, f_new
            if f_curr < best_f:
                best_x, best_y, best_f = x, y, f_curr

        T *= cooling
        history.append(best_f)

    return {"x": best_x, "y": best_y, "f": best_f, "history": history}


### 📝 แสดงการคำนวณด้วยมือ 2 Iterations — SA บน Rosenbrock (seed=0)

**ค่าเริ่มต้น:** $x_0=0.5478,\ y_0=0.0791$ (สุ่มจาก seed=0), $f(x_0,y_0)=5.0881$, $T_0=10$

**Iteration 1:**

สุ่ม candidate: $x'=0.8681,\ y'=0.1316$

$$f(x',y')=(1-0.8681)^2+100(0.1316-0.8681^2)^2=38.6968$$

$$\Delta = 38.6968-5.0881=33.6088 \ (\ge0 \Rightarrow \text{ต้องคำนวณความน่าจะเป็น})$$

$$P=e^{-\Delta/T}=e^{-33.6088/10}=e^{-3.361}=0.0347$$

สุ่มเลข uniform(0,1) ได้ค่ามากกว่า 0.0347 ⇒ **ปฏิเสธ (Reject)** — ยังคงอยู่ที่ $(0.5478,0.0791)$

ลดอุณหภูมิ: $T_1 = 10\times0.9=9$

**Iteration 2:**

สุ่ม candidate ใหม่: $x'=0.7286,\ y'=0.7311$

$$f(x',y')=(1-0.7286)^2+100(0.7311-0.7286^2)^2=4.0826$$

$$\Delta = 4.0826-5.0881=-1.0055 \ (<0 \Rightarrow \textbf{ยอมรับทันที})$$

อัปเดต: $x=0.7286,\ y=0.7311,\ f=4.0826$ — เป็นค่าดีที่สุดใหม่ (ดีกว่า $f_0=5.0881$)

ลดอุณหภูมิ: $T_2=9\times0.9=8.1$

> 📌 สังเกต: Iteration 1 คือตัวอย่างของการ**ปฏิเสธ**คำตอบที่แย่กว่ามาก (เพราะ $\Delta$ ใหญ่เกินไปเทียบกับ $T$)
> ส่วน Iteration 2 คือการ**ยอมรับ**คำตอบที่ดีขึ้น — นี่คือกลไกหลักของ SA ที่ทำให้มันต่างจาก pure hill-climbing
> (ซึ่งจะปฏิเสธคำตอบแย่กว่าเสมอ ไม่มีโอกาสหนีหลุม)


In [ ]:
# ตรวจคำตอบให้ตรงกับที่คำนวณมือด้านบน
_ = simulated_annealing(rosenbrock, [(-2, 2), (-1, 3)], n_iter=2, T0=10, cooling=0.9, step=0.5, seed=0, verbose=True)


### รันจริง ≥20 iterations พร้อม Best/Mean/Worst/Runtime/Convergence Curve

รัน **30 trials อิสระ** (คนละ random seed) แต่ละ trial ใช้ 500 iterations (สำหรับ Rosenbrock) และ 1000 iterations (สำหรับ Rastrigin
ที่ยากกว่าเพราะมีหลาย local minima) แล้วสรุปสถิติ


In [ ]:
def run_trials(algo_func, func, bounds, n_trials=30, **kwargs):
    """รันอัลกอริทึมหลาย trial แล้วเก็บสถิติ + convergence curve เฉลี่ย"""
    results, histories, runtimes = [], [], []
    for trial in range(n_trials):
        t0 = time.perf_counter()
        res = algo_func(func, bounds, seed=trial, **kwargs)
        runtimes.append(time.perf_counter() - t0)
        results.append(res["f"])
        histories.append(res["history"])

    max_len = max(len(h) for h in histories)
    histories_padded = np.array([h + [h[-1]] * (max_len - len(h)) for h in histories])

    stats = {
        "best": float(np.min(results)),
        "mean": float(np.mean(results)),
        "worst": float(np.max(results)),
        "std": float(np.std(results)),
        "avg_runtime_ms": float(np.mean(runtimes) * 1000),
        "mean_convergence": histories_padded.mean(axis=0),
        "all_final": results,
    }
    return stats


sa_rosenbrock = run_trials(simulated_annealing, rosenbrock, [(-2, 2), (-1, 3)],
                            n_trials=30, n_iter=500, T0=10, cooling=0.98, step=0.3)
sa_rastrigin = run_trials(simulated_annealing, rastrigin, [(-5, 5), (-5, 5)],
                           n_trials=30, n_iter=1000, T0=20, cooling=0.995, step=0.3)

print("SA on Rosenbrock (30 trials x 500 iters):")
print(f"  Best={sa_rosenbrock['best']:.5f}  Mean={sa_rosenbrock['mean']:.5f}  "
      f"Worst={sa_rosenbrock['worst']:.5f}  Std={sa_rosenbrock['std']:.5f}")
print(f"  Average runtime: {sa_rosenbrock['avg_runtime_ms']:.3f} ms/trial")

print("\nSA on Rastrigin (30 trials x 1000 iters):")
print(f"  Best={sa_rastrigin['best']:.5f}  Mean={sa_rastrigin['mean']:.5f}  "
      f"Worst={sa_rastrigin['worst']:.5f}  Std={sa_rastrigin['std']:.5f}")
print(f"  Average runtime: {sa_rastrigin['avg_runtime_ms']:.3f} ms/trial")
print("\n📌 สังเกต Std สูงและ Worst แย่กว่า Best มาก บน Rastrigin -> สะท้อนว่า SA ติดหลุม local minimum บ่อย")


## 3. Particle Swarm Optimization (PSO)

**แรงบันดาลใจ:** เลียนแบบ **ฝูงนก/ฝูงปลา** ที่หาอาหารร่วมกัน — แต่ละตัว (particle) จดจำจุดที่ดีที่สุดของตัวเอง
(`pbest`) และรู้จุดที่ดีที่สุดของทั้งฝูง (`gbest`) แล้วปรับทิศทางบินตามทั้งสองอย่าง

**สมการอัปเดต (สำคัญที่สุด)**

$$v_i \leftarrow \underbrace{w\,v_i}_{\text{inertia}} + \underbrace{c_1 r_1 (pbest_i-x_i)}_{\text{cognitive (ประสบการณ์ตัวเอง)}} + \underbrace{c_2 r_2 (gbest-x_i)}_{\text{social (ประสบการณ์ฝูง)}}$$

$$x_i \leftarrow x_i + v_i$$

โดย $r_1,r_2\sim U(0,1)$ สุ่มใหม่ทุกครั้ง, $w$=inertia weight, $c_1,c_2$=learning factors (มักตั้ง $\approx1.5$–$2$)

**เทียบกับ SA:** PSO เป็น **population-based** (ค้นหาพร้อมกันหลายจุด) ในขณะที่ SA เป็น **single-solution/trajectory**
(เดินคำตอบเดียวไปเรื่อยๆ) — ตามตารางในสไลด์ Lecture 6


In [ ]:
def pso(func, bounds, n_particles=20, n_iter=20, w=0.7, c1=1.5, c2=1.5, seed=None, verbose=False):
    rng = np.random.default_rng(seed)
    (xlo, xhi), (ylo, yhi) = bounds

    pos = np.column_stack([rng.uniform(xlo, xhi, n_particles), rng.uniform(ylo, yhi, n_particles)])
    vel = np.zeros((n_particles, 2))
    fitness = np.array([func(p[0], p[1]) for p in pos])

    pbest_pos = pos.copy()
    pbest_fit = fitness.copy()
    gbest_idx = np.argmin(pbest_fit)
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_fit = pbest_fit[gbest_idx]
    history = [gbest_fit]

    for it in range(1, n_iter + 1):
        r1 = rng.random((n_particles, 2))
        r2 = rng.random((n_particles, 2))
        vel = w * vel + c1 * r1 * (pbest_pos - pos) + c2 * r2 * (gbest_pos - pos)
        pos = pos + vel
        pos[:, 0] = np.clip(pos[:, 0], xlo, xhi)
        pos[:, 1] = np.clip(pos[:, 1], ylo, yhi)

        fitness = np.array([func(p[0], p[1]) for p in pos])
        improved = fitness < pbest_fit
        pbest_pos[improved] = pos[improved]
        pbest_fit[improved] = fitness[improved]

        if pbest_fit.min() < gbest_fit:
            gbest_idx = np.argmin(pbest_fit)
            gbest_pos = pbest_pos[gbest_idx].copy()
            gbest_fit = pbest_fit[gbest_idx]
        history.append(gbest_fit)

        if verbose and it <= 2:
            print(f"Iter {it}: gbest=({gbest_pos[0]:.4f},{gbest_pos[1]:.4f})  f_gbest={gbest_fit:.4f}")
            print(f"         particle fitness this round: {np.round(fitness,4)}\n")

    return {"x": gbest_pos[0], "y": gbest_pos[1], "f": gbest_fit, "history": history}


### 📝 แสดงการคำนวณด้วยมือ 2 Iterations — PSO บน Rosenbrock (2 particles, seed=0)

**ค่าเริ่มต้น (สุ่มจาก seed=0):**

| Particle | ตำแหน่งเริ่มต้น $(x,y)$ | $f$ |
|---|---|---|
| P1 | $(0.5478,\,-0.8361)$ | $129.31$ |
| P2 | $(-0.9209,\,-0.9339)$ | $321.19$ |

$pbest_1=(0.5478,-0.8361)$, $pbest_2=(-0.9209,-0.9339)$ (เท่ากับตำแหน่งเริ่มต้นของตัวเอง)

$gbest = pbest_1 = (0.5478,-0.8361)$ (ค่า $f$ ต่ำสุดในฝูง$=129.31$)

**Iteration 1 — อัปเดต P1 (ตัวที่เป็น gbest เอง):**

เพราะ $pos_1=pbest_1=gbest$ ทุกพจน์ในสมการ $(pbest_1-x_1)$ และ $(gbest-x_1)$ เป็น**ศูนย์** ⇒ $v_1=w(0)+c_1r_1(0)+c_2r_2(0)=0$

**P1 จึงไม่ขยับเลยในรอบนี้** — ยังอยู่ที่ $(0.5478,-0.8361)$, $f=129.31$ เท่าเดิม

**Iteration 1 — อัปเดต P2:** สุ่มได้ $r_1=(0.6066,0.7295),\ r_2=(0.8159,0.0027)$

$$v_2 = 0.7(0,0) + 1.5(0.6066,0.7295)\big[(0.5478,-0.8361)-(-0.9209,-0.9339)\big] + 1.5(0.8159,0.0027)\big[(0.5478,-0.8361)-(-0.9209,-0.9339)\big]$$

$$\approx (1.7974,\ 0.0004)$$

$$x_2^{new} = (-0.9209,-0.9339)+(1.7974,0.0004) = (0.8765,-0.9335)$$

$$f(0.8765,-0.9335) = 289.61 \quad(\text{ดีขึ้นจาก } 321.19 \text{ แต่ยังแย่กว่า } gbest=129.31)$$

เพราะ $289.61 < pbest_2^{old}=321.19$ ⇒ **อัปเดต** $pbest_2=289.61$ แต่ยังไม่ดีกว่า $gbest=129.31$ ⇒ **$gbest$ ไม่เปลี่ยน**

> 📌 สังเกต: particle ที่เป็น $gbest$ เองจะ "หยุดนิ่ง" ชั่วคราวถ้า inertia เริ่มที่ 0 (ตามที่ตั้งในโค้ด) — ส่วน particle อื่นจะถูก "ดึง" เข้าหา $gbest$ เสมอ ผ่านพจน์ social term นี่คือกลไกที่ทำให้ฝูงมารวมกันที่บริเวณดี


In [ ]:
# ตรวจคำตอบให้ตรงกับที่คำนวณมือด้านบน
_ = pso(rosenbrock, [(-2, 2), (-1, 3)], n_particles=2, n_iter=2, seed=0, verbose=True)


In [ ]:
pso_rosenbrock = run_trials(pso, rosenbrock, [(-2, 2), (-1, 3)],
                             n_trials=30, n_particles=20, n_iter=100)
pso_rastrigin = run_trials(pso, rastrigin, [(-5, 5), (-5, 5)],
                            n_trials=30, n_particles=30, n_iter=150)

print("PSO on Rosenbrock (30 trials x 100 iters, 20 particles):")
print(f"  Best={pso_rosenbrock['best']:.6f}  Mean={pso_rosenbrock['mean']:.6f}  "
      f"Worst={pso_rosenbrock['worst']:.6f}  Std={pso_rosenbrock['std']:.6f}")
print(f"  Average runtime: {pso_rosenbrock['avg_runtime_ms']:.3f} ms/trial")

print("\nPSO on Rastrigin (30 trials x 150 iters, 30 particles):")
print(f"  Best={pso_rastrigin['best']:.6f}  Mean={pso_rastrigin['mean']:.6f}  "
      f"Worst={pso_rastrigin['worst']:.6f}  Std={pso_rastrigin['std']:.6f}")
print(f"  Average runtime: {pso_rastrigin['avg_runtime_ms']:.3f} ms/trial")
print("\n📌 สังเกต: PSO ลู่เข้าใกล้ 0 ได้แม่นยำกว่า SA มากในจำนวน iteration ที่น้อยกว่า -> ข้อดีของ population-based search")


## 4. Convergence Curves

In [ ]:
def plot_convergence(stats_dict_by_algo, title, log_y=True):
    fig = go.Figure()
    for name, stats in stats_dict_by_algo.items():
        y = stats["mean_convergence"]
        fig.add_trace(go.Scatter(x=list(range(len(y))), y=y, mode="lines", name=name))
    fig.update_layout(title=title, xaxis_title="Iteration", yaxis_title="Best fitness so far (mean of trials)",
                       yaxis_type="log" if log_y else "linear", width=650, height=420)
    fig.show()

plot_convergence({"Simulated Annealing": sa_rosenbrock, "PSO": pso_rosenbrock},
                  "Convergence: Rosenbrock (lower is better)")
plot_convergence({"Simulated Annealing": sa_rastrigin, "PSO": pso_rastrigin},
                  "Convergence: Rastrigin (lower is better)")


## 5. เปรียบเทียบ SA vs PSO

| ประเด็น | Simulated Annealing | PSO |
|---|---|---|
| กลุ่ม | Single-solution (trajectory) | Population-based |
| กลไกหนี local optimum | ยอมรับคำตอบแย่กว่าแบบ probabilistic ($T$ ควบคุม) | ใช้หลายจุดค้นหาพร้อมกัน + social attraction |
| Parameter หลัก | $T_0$, cooling rate, step size | จำนวน particles, $w$, $c_1$, $c_2$ |
| ความเร็วต่อ iteration | เร็ว (คำนวณจุดเดียว) | ช้ากว่า (คำนวณหลาย particle) |
| คุณภาพคำตอบ (จากผลทดลองข้างบน) | Std สูงกว่า โดยเฉพาะบน Rastrigin | ลู่เข้า global optimum แม่นยำกว่าในภาพรวม |
| เหมาะกับ | ปัญหาที่ evaluate function แพง (คำนวณจุดเดียวต่อรอบ) | ปัญหาที่ evaluate function ถูก และมี landscape ซับซ้อนมาก |

**ทำไม PSO เอาชนะ SA ในการทดลองนี้:** เพราะ PSO มี "สายตา" หลายจุดพร้อมกัน (20-30 particles) ที่ช่วยกันสำรวจ
ในขณะที่ SA มีตัวแทนเดียวที่ต้องอาศัยการสุ่ม step และ cooling schedule ที่ต้อง tune ให้ดีมาก — แต่ข้อแลกเปลี่ยนคือ
PSO ใช้จำนวนการประเมินฟังก์ชัน (function evaluations) มากกว่า SA หลายเท่าต่อ iteration (20-30 เท่า)
ถ้าเทียบที่ **จำนวน function evaluations เท่ากัน** ผลอาจใกล้เคียงกันมากขึ้น


## 6. ตัวอย่างการนำไปใช้จริง (Real-world Optimization Problem)

**Hyperparameter Tuning ของ Machine Learning Model**

เวลาเทรนโมเดล เช่น Neural Network หรือ Random Forest เราต้องเลือกค่า hyperparameter (learning rate, จำนวน layer,
regularization strength ฯลฯ) ที่ทำให้ **validation loss ต่ำที่สุด** — แต่ฟังก์ชัน `loss(hyperparameters)` นี้:

- **ไม่มีสูตรปิด (closed-form)** — ต้องเทรนโมเดลจริงเพื่อรู้ค่า loss (evaluate ฟังก์ชันแพงมาก อาจใช้เวลาเป็นนาที/ชั่วโมงต่อครั้ง)
- **ไม่ convex** — มี local minima หลายจุด เหมือน Rastrigin
- **หา gradient ไม่ได้ตรงๆ** (hyperparameter หลายตัวเป็น discrete เช่น จำนวน layer)

จึงเหมาะกับ metaheuristic แบบที่ทำในไฟล์นี้พอดี:

- ถ้า evaluate model **แพงมาก** (เช่น เทรน deep learning ใหญ่ๆ) → เลือก **SA** หรือ **Bayesian Optimization**
  เพราะประเมินทีละจุด ประหยัด compute
- ถ้า evaluate model **ถูก** (โมเดลเล็ก เทรนเร็ว) และมี compute ขนาน (GPU/cluster หลายตัว) → เลือก **PSO** หรือ **Genetic Algorithm**
  เพราะประเมินหลายจุดพร้อมกันได้ (แต่ละ particle เทรนโมเดลแยกกันบนเครื่องคนละตัว)

**ตัวอย่างอื่นที่ใช้แนวคิดเดียวกัน:** การจัดตารางการผลิตในโรงงาน (production scheduling), การออกแบบเส้นทางขนส่ง
(vehicle routing), การปรับ portfolio การลงทุนภายใต้ความเสี่ยง — ทุกกรณีมีจุดร่วมคือ **objective function ซับซ้อน/ไม่ smooth
ที่วิธีเชิงคณิตศาสตร์ตรงๆ (Simplex, Gradient Descent) ใช้ไม่ได้** จึงต้องพึ่ง metaheuristic


## 7. ลองปรับพารามิเตอร์ของคุณเอง

แก้ค่าพารามิเตอร์ด้านล่าง เช่น จำนวน particles, cooling rate, จำนวน trials แล้วรันใหม่เพื่อดูผลกระทบต่อ
Best/Mean/Worst และ convergence curve


In [ ]:
# ==== ปรับพารามิเตอร์ตรงนี้ ====
custom_sa = run_trials(simulated_annealing, rastrigin, [(-5, 5), (-5, 5)],
                        n_trials=20, n_iter=500, T0=15, cooling=0.99, step=0.4)
custom_pso = run_trials(pso, rastrigin, [(-5, 5), (-5, 5)],
                         n_trials=20, n_particles=40, n_iter=100)

print("Custom SA :", f"Best={custom_sa['best']:.5f}  Mean={custom_sa['mean']:.5f}  Worst={custom_sa['worst']:.5f}")
print("Custom PSO:", f"Best={custom_pso['best']:.5f}  Mean={custom_pso['mean']:.5f}  Worst={custom_pso['worst']:.5f}")

plot_convergence({"SA (custom)": custom_sa, "PSO (custom)": custom_pso}, "Custom parameters on Rastrigin")
